# Mistral-7B LoRA — 5-Fold K-Fold Ensemble (Experiment 5)

**Task:** Binary classification of political statements as true (0) or false (1)  
**Model:** `mistralai/Mistral-7B-v0.1` — 7B parameters, loaded in 4-bit NF4 (~3.5 GB VRAM)  
**Technique:** LoRA (Low-Rank Adaptation) — only ~4M of 7B parameters are trained  
**Metric:** Macro F1

## Why a 7B model for 8,950 samples?

Mistral-7B has learned world knowledge from hundreds of billions of tokens. Even though our dataset is small, the model already knows about political figures, parties, and the typical language patterns of credible vs misleading claims. LoRA lets us adapt this knowledge with minimal compute — without LoRA, fine-tuning 7B parameters at FP32 would require ~112 GB VRAM.

## Experiment results

| Experiment | Model | Strategy | Holdout Macro F1 | ROC-AUC |
|---|---|---|---|---|
| Exp 2 | DeBERTa-v3-small | K-Fold ×5 | 0.6393 | 0.6954 |
| Exp 4a | DeBERTa-v3-base | K-Fold ×5 | 0.6397 | 0.7004 |
| **Exp 5 (this)** | **Mistral-7B LoRA** | **K-Fold ×5** | **0.6553** | **0.7240** |

This is the **project's best standalone result** — a +0.016 F1 / +0.029 AUC improvement over DeBERTa-v3-base. The jump reflects the 40× parameter count difference and Mistral's stronger pre-training on politically diverse text.

## What is LoRA?

LoRA (Hu et al., 2021) decomposes each weight update into a low-rank product:

```
W_adapted = W_frozen + (B × A)  where  A ∈ ℝ^{r×d_in},  B ∈ ℝ^{d_out×r}
```

For r=8 and d=4096 (Mistral attention dim): `W_frozen` has 16M params — **frozen**. The LoRA pair `(A, B)` has 2×8×4096 = 65,536 params — **trained**. Applied to q/k/v/o projections across all 32 layers: ~4M trainable params total.

The original weights never change — only the small adapter matrices are updated. This makes training 16× faster than full fine-tuning and the adapters (a few MB) can be saved independently from the 14 GB base model weights.

## What is 4-bit NF4 quantization?

NF4 (NormalFloat4, Dettmers et al., 2023) stores each weight as a 4-bit index into a lookup table optimised for normally-distributed neural network weights. The 7B FP32 model (28 GB) becomes ~3.5 GB in memory. Compute happens in BF16 via dequantize-on-the-fly during forward pass — accuracy loss is negligible for fine-tuning tasks.

**Note:** Fold 1 in the first LoRA run (`Lora 1`) collapsed (val macro_f1 ~0.40) because `LR=2e-4` was too high — the adapter updates overshot and the classification head diverged. This notebook uses `LR=5e-5` (the fixed version, `Lora 3`), which kept all 5 folds healthy.

In [ ]:
# pip install peft bitsandbytes accelerate  (one-time install)
from datetime import datetime
from pathlib import Path
import gc
import sys
from time import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)

## Environment & Configuration

**Key differences from DeBERTa experiments:**

| Parameter | DeBERTa-v3-small | **Mistral-7B LoRA** | Why different |
|---|---|---|---|
| `BATCH_SIZE` | 16 | **4** | BF16 activations + 7B params; gradient checkpointing recovers memory |
| `GRAD_ACCUM` | — | **4** | Effective batch = 16, matching DeBERTa experiments for a fair comparison |
| `LR` | 2e-5 | **5e-5** | LoRA adapters start at zero and can tolerate a higher LR than full fine-tuning |
| `LORA_R` | — | **8** | Rank of decomposition; ~4M trainable params total |
| `LORA_ALPHA` | — | **16** | Scaling factor: effective update magnitude = `(alpha/r) × adapter_update = 2×` |
| `FREEZE_EPOCHS` | 0 | **n/a** | All backbone weights are frozen by construction — LoRA only trains the adapter matrices |
| `USE_AMP` | False (explicit) | **n/a** | Compute dtype is set inside `BitsAndBytesConfig`; no separate autocast needed |

In [ ]:
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_DIR   = Path("/kaggle/input/truth-classifier-nlp")
    OUTPUT_DIR = Path("/kaggle/working/models/transformer_lora_kfold")
else:
    def _find_root(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "data" / "train.csv").exists():
                return p
        raise FileNotFoundError("Cannot find project root")
    _root      = _find_root(Path.cwd())
    DATA_DIR   = _root / "data"
    OUTPUT_DIR = _root / "models" / "transformer_lora_kfold"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME   = "mistralai/Mistral-7B-v0.1"
MAX_LENGTH   = 128
BATCH_SIZE   = 4
GRAD_ACCUM   = 4      # effective batch = 16
EPOCHS       = 3
LR           = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
K_FOLDS      = 5

LORA_R              = 8
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

CLASS_WEIGHTS = [1.42, 0.77]
THRESHOLD     = 0.5
SEED          = 42

create_kaggle_csv = True
model_slug        = "mistral-7b-lora3-kfold"

NUM_WORKERS = 0 if sys.platform == "win32" else 2

torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM : {vram:.1f} GB")
    print(f"  Note : 7B model weights ~3.5 GB in NF4; activations + adapters fit in {vram:.0f} GB")

_script_start = time()
def _now() -> str:
    return datetime.now().strftime("%H:%M:%S")

## Input Formatter

Identical to all prior transformer notebooks — prepends `speaker | party | subject` as raw tokens. The same format was used in Exp 1b/2/4 so results are comparable.

One consideration with a 7B causal decoder: the prefix tokens (`speaker: X | party: Y | ...`) appear **before** the statement. Mistral's attention is causal (each token attends only to previous tokens), so earlier prefix tokens can influence later statement tokens — but not vice versa. This is actually a natural fit: the model reads context first, then the claim.

In [ ]:
def format_input(speaker: str, party: str, subject: str, statement: str) -> str:
    def _clean(val) -> str:
        if pd.isna(val) or str(val).strip() == "":
            return "unknown"
        return str(val).strip().lower()
    primary_subject = _clean(subject).split(",")[0].strip()
    return (
        f"speaker: {_clean(speaker)} | "
        f"party: {_clean(party)} | "
        f"subject: {primary_subject} | "
        f"{statement}"
    )

## Data Loading & Outer Split

Same 80/20 outer holdout split with `random_state=42` as all other experiments. The holdout is tokenised once and reused across folds.

In [ ]:
print(f"[{_now()}] Loading data...")
df = pd.read_csv(DATA_DIR / "train.csv")
print(f"  Rows: {len(df):,}  |  Labels: {df['label'].value_counts().to_dict()}")

all_texts  = df.apply(
    lambda r: format_input(
        r["speaker"], r["party_affiliation"], r["subject"], r["statement"]
    ),
    axis=1,
).tolist()
all_labels = np.array(df["label"].tolist())
all_idx    = np.arange(len(df))

tv_idx, ho_idx = train_test_split(
    all_idx, test_size=0.2, random_state=SEED, stratify=all_labels
)
tv_labels = all_labels[tv_idx]
ho_labels = all_labels[ho_idx]
X_ho      = [all_texts[i] for i in ho_idx]

print(f"  Trainval: {len(tv_idx):,}   Holdout: {len(ho_idx):,}")

if create_kaggle_csv:
    df_test    = pd.read_csv(DATA_DIR / "test_nolabel.csv")
    test_texts = df_test.apply(
        lambda r: format_input(
            r["speaker"], r["party_affiliation"], r["subject"], r["statement"]
        ),
        axis=1,
    ).tolist()
    print(f"  Test rows: {len(test_texts):,}")

## Tokenizer — Left-Padding for Causal Decoders

Mistral-7B is a **decoder-only** (causal) model: each token attends only to tokens that come before it. The classification head reads the last token's hidden state — so padding matters:

```
Right-pad (BERT style): [speaker: obama | ... statement] [PAD] [PAD] → last token is PAD
Left-pad  (correct):    [PAD] [PAD] [speaker: obama | ... statement] → last token is real
```

With right-padding, the `[CLS]`-equivalent position would be a PAD token, which has no informative hidden state. Left-padding ensures the final real token carries the fully-attended sequence representation.

**`pad_token = eos_token`**: Mistral's vocabulary has no dedicated `[PAD]` token. Using the end-of-sequence token as padding is the standard workaround — the model has seen EOS in pre-training as a sentence boundary marker, so it is a semantically neutral pad.

In [ ]:
print(f"[{_now()}] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token    = tokenizer.eos_token   # Mistral has no [PAD]
tokenizer.padding_side = "left"                # causal decoder: pad on left

print(f"  Vocab size    : {tokenizer.vocab_size:,}")
print(f"  Pad token     : '{tokenizer.pad_token}'  (id={tokenizer.pad_token_id})")
print(f"  Padding side  : {tokenizer.padding_side}")

# Show a formatted example
example_row  = df.iloc[0]
example_text = format_input(
    example_row["speaker"], example_row["party_affiliation"],
    example_row["subject"],  example_row["statement"]
)
example_enc = tokenizer(example_text, return_tensors="pt")
print(f"  Example text  : {example_text[:100]}...")
print(f"  Token count   : {example_enc['input_ids'].shape[1]}  (MAX_LENGTH={MAX_LENGTH})")


class StatementDataset(Dataset):
    def __init__(self, texts: list[str], labels):
        self.enc    = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        return {k: v[idx] for k, v in self.enc.items()}, self.labels[idx]


print(f"\n  Tokenizing holdout ({len(X_ho):,} rows)...")
holdout_ds     = StatementDataset(X_ho, ho_labels)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE * 2,
                            shuffle=False, num_workers=NUM_WORKERS)
print(f"  Holdout tokenized")

## Model Factory — 4-Bit Quantization + LoRA

The model is loaded fresh per fold inside `_build_model()`. Each call:

### Step 1 — 4-bit NF4 quantization

```python
BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — optimised for normally-distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16, # dequantize to BF16 for matmuls
    bnb_4bit_use_double_quant=True,     # quantize the quantization constants too (-0.5 GB)
)
```

The 7B FP32 model (28 GB) loads as ~3.5 GB. `device_map={"":0}` places all layers on GPU 0 — required for quantized models; `.to(device)` does not work with bitsandbytes.

### Step 2 — Gradient checkpointing

`prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)` enables gradient checkpointing on the quantized backbone. Instead of storing all intermediate activations (which would overflow VRAM), only a subset are cached — the others are recomputed during the backward pass. This trades ~30% extra compute for ~4× VRAM savings on activations.

### Step 3 — LoRA adapter injection

```python
LoraConfig(
    r=8,          # adapter rank — controls trainable param count
    lora_alpha=16, # scaling: effective update = (alpha/r) × BA = 2 × BA
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # all 4 attention projections
    modules_to_save=["score"],  # the classification head — fully trained, not via LoRA
)
```

`get_peft_model` freezes all base model weights and injects trainable `(A, B)` matrices alongside each targeted layer.

### Step 4 — Near-zero classification head init

The default random head init (`std ≈ 0.02`) produces logits of magnitude 10–17 against Mistral's unnormalized hidden states (which are much larger than BERT's layer-normed `[CLS]`). These large logits cause batch-0 losses of 10–17 instead of `log(2) ≈ 0.69`, which overwhelms the LoRA adapter gradients in epoch 1 and can cause divergence. Reinitialising the score head with `std=0.01` keeps the initial loss near `log(2)` while keeping weights non-zero so gradients still flow.

In [ ]:
def _build_model() -> nn.Module:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        quantization_config=bnb_config,
        device_map={"" : 0},          # required for quantized models — .to(device) won't work
        pad_token_id=tokenizer.eos_token_id,
    )
    model.config.use_cache = False    # disable KV-cache — incompatible with gradient checkpointing

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        modules_to_save=["score"],    # classification head — fully trained alongside adapters
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # Near-zero init for classification head to keep batch-0 loss near log(2)
    with torch.no_grad():
        for name, param in model.named_parameters():
            if "score" in name and param.requires_grad:
                nn.init.normal_(param, mean=0.0, std=0.01)
                break

    return model

## Training Helpers — Gradient Accumulation

With `BATCH_SIZE=4` (GPU memory limit) and `GRAD_ACCUM=4`, the effective batch size is 16 — matching all DeBERTa experiments.

**How gradient accumulation works:**
```
for batch 0:  loss = criterion(...) / 4  →  loss.backward()  (grads accumulate)
for batch 1:  loss = criterion(...) / 4  →  loss.backward()  (grads accumulate)
for batch 2:  loss = criterion(...) / 4  →  loss.backward()  (grads accumulate)
for batch 3:  loss = criterion(...) / 4  →  loss.backward()  →  optimizer.step()  →  zero_grad()
```

Dividing loss by `GRAD_ACCUM` before backward ensures the accumulated gradient is the **mean** over 16 samples (not the sum). Without this division, the effective LR would scale with accumulation steps.

**Scheduler steps on optimizer updates, not batches:** `scheduler.step()` is called only when the optimizer steps (every `GRAD_ACCUM` batches). The warmup schedule uses `total_updates = (batches_per_epoch // GRAD_ACCUM) × EPOCHS` so the LR ramp is correctly proportioned to actual parameter updates.

In [ ]:
loss_weights = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(device)
criterion    = nn.CrossEntropyLoss(weight=loss_weights)


def train_epoch(model, loader, optimizer, scheduler) -> float:
    model.train()
    total_loss    = 0.0
    optimizer.zero_grad()

    for batch_idx, (inputs, labs) in enumerate(loader):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labs   = labs.to(device)
        logits = model(**inputs).logits
        loss   = criterion(logits, labs) / GRAD_ACCUM  # divide so accumulated grad = mean
        loss.backward()

        is_update = (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(loader)
        if is_update:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()   # step on optimizer update, not on batch
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM  # undo the /GRAD_ACCUM for logging
        if batch_idx == 0:
            print(f"    Batch 0 — loss={loss.item() * GRAD_ACCUM:.4f}")
        if (batch_idx + 1) % 100 == 0:
            print(f"    Batch {batch_idx+1}/{len(loader)}  avg_loss={total_loss/(batch_idx+1):.4f}")

    return total_loss / len(loader)


@torch.no_grad()
def predict_proba(model, loader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    all_logits, all_labels_list, total_loss = [], [], 0.0
    for inputs, labs in loader:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        logits = model(**inputs).logits
        loss   = criterion(logits, labs.to(device))
        total_loss      += loss.item()
        all_logits.append(logits.float().cpu())
        all_labels_list.append(labs)
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels_list).numpy()
    proba  = torch.softmax(logits, dim=-1)[:, 1].numpy()
    return total_loss / len(loader), proba, labels


@torch.no_grad()
def predict_proba_texts(model, texts: list[str]) -> np.ndarray:
    model.eval()
    enc = tokenizer(
        texts, truncation=True, padding="max_length",
        max_length=MAX_LENGTH, return_tensors="pt"
    )
    all_proba = []
    _bsz = BATCH_SIZE * 2
    for i in range(0, len(texts), _bsz):
        batch  = {k: v[i:i + _bsz].to(device) for k, v in enc.items()}
        logits = model(**batch).logits
        all_proba.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(all_proba)

## K-Fold Training Loop

The k-fold structure mirrors `DeBERTa_KFold.ipynb` with three important differences:

1. **`torch.manual_seed(SEED + fold_k)`** before `_build_model()` — the LoRA adapter matrices `A` are initialised with a random normal distribution; seeding per fold ensures the five folds start with different (but reproducible) adapter initialisations, increasing ensemble diversity.

2. **Checkpoint saves only trainable parameters** — not the full 7B model state dict. The LoRA adapters + classification head are ~16 MB total; saving the full quantized model would be ~3.5 GB per fold. The adapter-only checkpoint is sufficient because the frozen base weights are always reloaded from HuggingFace at the start of the next fold.

3. **Adapter restore is in-place**: after training, best adapter weights are copied back into the model tensors with `p.data.copy_()` — the quantized base weights are not re-downloaded. Then `model.save_pretrained(adapter_dir)` writes the PEFT adapter config and weights (a few MB), which the extract script can reload later.

4. **`gc.collect()` in addition to `cuda.empty_cache()`** — the 7B quantized model has more non-tensor Python objects (bitsandbytes state, quantization tables) that Python's garbage collector needs to free. Without `gc.collect()`, VRAM usage can drift upward across folds.

In [ ]:
print(f"[{_now()}] Starting {K_FOLDS}-fold LoRA training")
print(f"  K={K_FOLDS}  EPOCHS={EPOCHS}  BATCH={BATCH_SIZE}  GRAD_ACCUM={GRAD_ACCUM}")
print(f"  Effective batch = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LoRA: r={LORA_R}  alpha={LORA_ALPHA}  targets={LORA_TARGET_MODULES}")
print(f"  Expected runtime: ~15-25 min per fold")

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

oof_proba        = np.zeros(len(tv_idx))
ho_proba_folds   = []
test_proba_folds = []
fold_log         = []

for fold_k, (tr_rel, val_rel) in enumerate(skf.split(tv_idx, tv_labels)):
    print(f"\n{'='*70}")
    print(f"  FOLD {fold_k + 1}/{K_FOLDS}  [{_now()}]")
    print(f"{'='*70}")

    tr_abs  = tv_idx[tr_rel]
    val_abs = tv_idx[val_rel]

    X_tr_f  = [all_texts[i] for i in tr_abs]
    y_tr_f  = all_labels[tr_abs]
    X_val_f = [all_texts[i] for i in val_abs]
    y_val_f = all_labels[val_abs]

    print(f"  Train: {len(X_tr_f):,}   Val: {len(X_val_f):,}")

    _t0 = time()
    train_ds = StatementDataset(X_tr_f, y_tr_f)
    val_ds   = StatementDataset(X_val_f, y_val_f)
    print(f"  Tokenized in {time()-_t0:.1f}s")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,
                              num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                              num_workers=NUM_WORKERS)

    # Fresh quantized model + new LoRA adapters for this fold
    torch.manual_seed(SEED + fold_k)  # different adapter init per fold
    print(f"  Loading {MODEL_NAME}...")
    _t0   = time()
    model = _build_model()
    print(f"  Model loaded in {time()-_t0:.1f}s")

    # Scheduler uses optimizer update count (not batch count)
    update_steps_per_epoch = len(train_loader) // GRAD_ACCUM
    total_updates          = update_steps_per_epoch * EPOCHS
    warmup_updates         = int(total_updates * WARMUP_RATIO)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_updates, total_updates)
    print(f"  Trainable tensors: {len(trainable_params)}  LR={LR:.1e}  "
          f"total_updates={total_updates}  warmup={warmup_updates}")

    best_val_f1  = -1.0
    best_ckpt_pt = OUTPUT_DIR / f"fold{fold_k+1}-best.pt"

    for epoch in range(1, EPOCHS + 1):
        _t = time()
        print(f"\n  --- Epoch {epoch}/{EPOCHS} ---  [{_now()}]")
        train_loss = train_epoch(model, train_loader, optimizer, scheduler)
        val_loss, val_proba_ep, val_labels_ep = predict_proba(model, val_loader)

        val_pred = (val_proba_ep >= 0.5).astype(int)
        val_f1   = f1_score(val_labels_ep, val_pred, average="macro", zero_division=0)
        val_auc  = roc_auc_score(val_labels_ep, val_proba_ep)

        print(
            f"  Fold {fold_k+1} Ep {epoch}/{EPOCHS}  "
            f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
            f"val_macro_f1={val_f1:.4f}  val_roc_auc={val_auc:.4f}  "
            f"({time()-_t:.1f}s)"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            # Save only trainable params (~16 MB) — not the full 3.5 GB quantized model
            torch.save(
                {n: p.data.cpu() for n, p in model.named_parameters() if p.requires_grad},
                best_ckpt_pt,
            )
            print(f"    New best val macro_f1={best_val_f1:.4f} — adapter checkpoint saved")

    print(f"\n  Fold {fold_k+1} best val macro_f1={best_val_f1:.4f}")
    fold_log.append({"fold": fold_k + 1, "best_val_f1": best_val_f1,
                     "n_train": len(X_tr_f), "n_val": len(X_val_f)})

    # Restore best adapter weights in-place into the loaded quantized model
    best_state = torch.load(best_ckpt_pt, map_location="cpu")
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad and n in best_state:
                p.data.copy_(best_state[n].to(p.device))

    # Save PEFT adapter directory (for extract script / future inference)
    adapter_dir = OUTPUT_DIR / f"fold{fold_k+1}-adapter"
    model.save_pretrained(str(adapter_dir))
    print(f"  PEFT adapter saved: {adapter_dir.name}")

    # Collect OOF probas for this fold's val rows
    _, val_proba_best, _ = predict_proba(model, val_loader)
    oof_proba[val_rel]   = val_proba_best

    # Holdout probas from this fold
    _, ho_proba_fold, _ = predict_proba(model, holdout_loader)
    ho_proba_folds.append(ho_proba_fold)

    # Test probas from this fold
    if create_kaggle_csv:
        test_proba_folds.append(predict_proba_texts(model, test_texts))
        print(f"  Test inference done ({len(test_texts):,} rows)")

    # Thorough cleanup: gc.collect() frees bitsandbytes state not caught by cuda.empty_cache()
    del model, optimizer, scheduler, best_state
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU memory cleared")

print(f"\n[{_now()}] K-fold training complete")

## Per-Fold Summary

With LoRA, fold stability is more sensitive to the learning rate than with full fine-tuning — a rate that works for 4 folds can cause one fold to diverge if its val partition is particularly hard. This is what happened in the first LoRA run (`LR=2e-4`): fold 1 collapsed to val macro_f1 ~0.40 while folds 2–5 were healthy. The fix in this version (`LR=5e-5`) keeps all 5 folds stable.

In [ ]:
fold_summary = pd.DataFrame(fold_log).set_index("fold")
print("Per-fold results:")
display(fold_summary.round(4))

fig_folds, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    [f"Fold {r}" for r in fold_summary.index],
    fold_summary["best_val_f1"],
    color="darkorange", edgecolor="white"
)
mean_f1 = fold_summary["best_val_f1"].mean()
ax.axhline(mean_f1, color="red", linestyle="--",
           label=f"Mean = {mean_f1:.4f}")
for bar, val in zip(bars, fold_summary["best_val_f1"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, max(fold_summary["best_val_f1"]) + 0.06)
ax.set_ylabel("Best Val Macro F1")
ax.set_title(f"Mistral-7B LoRA — Per-Fold Best Val Macro F1  (K={K_FOLDS}, LR={LR:.0e})")
ax.legend()
plt.tight_layout()
plt.show()

## OOF Evaluation & Threshold Tuning

OOF probas from all 5 folds are now assembled into a 7,160-row array — one probability per trainval row, each predicted by a model that never saw it. Threshold is tuned on this full OOF array.

**Note on OOF quality:** if fold 1 had collapsed (Lora 1 scenario), its 1,432 OOF rows would have near-random probabilities, dragging down the OOF macro_f1 and forcing the threshold sweep to compensate. This is why the standalone Lora 1 result (0.6553) came from directly evaluating healthy folds 2–5 on the holdout, while the OOF-based threshold was less reliable.

In [ ]:
print(f"[{_now()}] OOF evaluation (trainval, N={len(tv_idx):,})")

oof_auc = roc_auc_score(tv_labels, oof_proba)
print(f"  OOF ROC-AUC (threshold-independent): {oof_auc:.4f}")
print(f"  OOF proba range: [{oof_proba.min():.4f}, {oof_proba.max():.4f}]")

grid = np.arange(0.20, 0.76, 0.01)
oof_scores = {
    round(float(t), 2): f1_score(
        tv_labels, (oof_proba >= t).astype(int),
        average="macro", zero_division=0
    )
    for t in grid
}
best_t = max(oof_scores, key=oof_scores.get)

print(f"\n  {'threshold':>10}   macro_f1")
for t, s in oof_scores.items():
    print(f"  {t:>10.2f}   {s:.4f}{'  <--' if t == best_t else ''}")
print(f"\n  Best OOF threshold: {best_t:.2f}  (OOF macro_f1={oof_scores[best_t]:.4f})")
THRESHOLD = best_t

fig_thr, ax_thr = plt.subplots(figsize=(10, 4))
ax_thr.plot(list(oof_scores.keys()), list(oof_scores.values()),
            color="darkorange", marker="o", markersize=3, label="OOF Macro F1")
ax_thr.axvline(best_t, color="red", linestyle="--",
               label=f"Best = {best_t:.2f}  (F1={oof_scores[best_t]:.4f})")
ax_thr.axvline(0.5, color="gray", linestyle=":", alpha=0.8, label="Default 0.5")
ax_thr.set_xlabel("Classification Threshold")
ax_thr.set_ylabel("OOF Macro F1")
ax_thr.set_title("OOF Threshold Sweep — Mistral-7B LoRA (full trainval)")
ax_thr.legend()
plt.tight_layout()
plt.show()

oof_pred = (oof_proba >= THRESHOLD).astype(int)
oof_f1   = f1_score(tv_labels, oof_pred, average="macro", zero_division=0)
print(f"\n  OOF macro_f1={oof_f1:.4f}  OOF roc_auc={oof_auc:.4f}")
print(f"  Using THRESHOLD = {THRESHOLD:.2f}")

## Ensemble Holdout Evaluation

The 5-fold ensemble prediction is the mean of 5 independent holdout probability arrays. With a 7B model, each fold has higher capacity and better-calibrated probabilities than the DeBERTa models — the ensemble gain from averaging is therefore smaller in relative terms, but the absolute probabilities are more informative.

In [ ]:
print(f"[{_now()}] Ensemble holdout evaluation  (threshold={THRESHOLD:.2f})")

ho_proba_ensemble = np.mean(ho_proba_folds, axis=0)
ho_pred = (ho_proba_ensemble >= THRESHOLD).astype(int)

holdout_metrics = {
    "roc_auc":      roc_auc_score(ho_labels, ho_proba_ensemble),
    "pr_auc":       average_precision_score(ho_labels, ho_proba_ensemble),
    "macro_f1":     f1_score(ho_labels, ho_pred, average="macro", zero_division=0),
    "f1":           f1_score(ho_labels, ho_pred, zero_division=0),
    "precision":    precision_score(ho_labels, ho_pred, zero_division=0),
    "recall":       recall_score(ho_labels, ho_pred, zero_division=0),
    "accuracy":     accuracy_score(ho_labels, ho_pred),
    "mcc":          matthews_corrcoef(ho_labels, ho_pred),
    "balanced_acc": balanced_accuracy_score(ho_labels, ho_pred),
}
cm = confusion_matrix(ho_labels, ho_pred)

print("\nHoldout results (5-fold ensemble):")
for name, val in holdout_metrics.items():
    print(f"  {name:15s}: {val:.4f}")
print(f"\n{classification_report(ho_labels, ho_pred, target_names=['True (0)', 'False (1)'])}")

## Evaluation Plots

In [ ]:
fpr, tpr, _      = roc_curve(ho_labels, ho_proba_ensemble)
prec_c, rec_c, _ = precision_recall_curve(ho_labels, ho_proba_ensemble)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(fpr, tpr, color="darkorange",
             label=f"ROC-AUC = {holdout_metrics['roc_auc']:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.6)
axes[0].set(title=f"ROC Curve — {model_slug} ensemble (holdout)",
            xlabel="False Positive Rate", ylabel="True Positive Rate")
axes[0].legend()

axes[1].plot(rec_c, prec_c, color="darkorange",
             label=f"PR-AUC = {holdout_metrics['pr_auc']:.4f}")
axes[1].set(title="Precision-Recall Curve (holdout)",
            xlabel="Recall", ylabel="Precision")
axes[1].legend()

im = axes[2].imshow(cm, interpolation="nearest", cmap="Oranges")
axes[2].set_title("Confusion Matrix (holdout, ensemble)")
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["True (0)", "False (1)"])
axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(["True (0)", "False (1)"])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
fig.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

## Saving OOF Probas

The OOF CSV follows the same format as `DeBERTa_KFold.ipynb` and can be fed as a late-fusion column into the stacking ensemble. Mistral's OOF probas carry a stronger signal than DeBERTa's — the stacking meta-LR should assign them an even higher coefficient than the 1.68 assigned to DeBERTa-v3-small OOF in Experiment 3.

In [ ]:
print(f"[{_now()}] Saving OOF probas")

oof_df = pd.DataFrame({
    "idx":       tv_idx,
    "oof_proba": oof_proba,
    "label":     tv_labels,
})
oof_path = OUTPUT_DIR / f"{model_slug}-oof.csv"
oof_df.to_csv(oof_path, index=False)
print(f"  OOF saved: {oof_path}  ({len(oof_df):,} rows)")
print(f"  OOF proba range : [{oof_proba.min():.4f}, {oof_proba.max():.4f}]")
print(f"  OOF macro_f1    : {oof_f1:.4f}")
print(f"  OOF ROC-AUC     : {oof_auc:.4f}")

assert len(oof_df) == len(tv_idx), "OOF length mismatch"
assert not np.any(oof_proba == 0), "Some OOF probas were never written — check fold coverage"
print("  OOF coverage check passed")

display(oof_df.head())

## Kaggle Submission

In [ ]:
if create_kaggle_csv:
    print(f"[{_now()}] Creating Kaggle submission (ensemble of {K_FOLDS} folds)")

    test_proba_ensemble = np.mean(test_proba_folds, axis=0)
    test_pred = (test_proba_ensemble >= THRESHOLD).astype(int)

    sub_dir = Path("/kaggle/working") if IS_KAGGLE else (_root / "submissions")
    sub_dir.mkdir(exist_ok=True)
    sub_path = sub_dir / f"submission-{model_slug}-{datetime.now().strftime('%Y%m%d-%H%M')}.csv"
    pd.DataFrame({"id": df_test["id"], "label": test_pred}).to_csv(sub_path, index=False)
    print(f"  Submission saved: {sub_path}  ({len(test_pred):,} rows)")
    print(f"  Predicted class distribution: {pd.Series(test_pred).value_counts().to_dict()}")

print(f"\n[DONE] Total time: {time()-_script_start:.1f}s  [{_now()}]")

## Summary

### Results

| Model | Trainable params | Holdout Macro F1 | ROC-AUC | Runtime (RTX 5070) |
|---|---|---|---|---|
| DeBERTa-v3-small (K-Fold) | 86M (full) | 0.6393 | 0.6954 | ~30 min |
| DeBERTa-v3-base (K-Fold) | 184M (full) | 0.6397 | 0.7004 | ~50 min |
| **Mistral-7B LoRA (K-Fold)** | **~4M of 7B** | **0.6553** | **0.7240** | **~90–120 min** |

### Key findings

1. **LoRA on a 7B model beats full fine-tuning of 86M and 184M models** — the gain comes from Mistral's richer pre-training, not from more trainable parameters (LoRA only trains ~4M)

2. **Learning rate is the critical hyperparameter for LoRA stability** — `LR=2e-4` caused fold 1 to collapse; `LR=5e-5` kept all folds healthy. LoRA adapters start at zero and are sensitive to early gradient magnitudes

3. **Near-zero head init is essential** — Mistral's unnormalized hidden states produce large initial logits without it, destabilizing adapter training in epoch 1

4. **Left-padding is required** — causal decoders must read the last real token for classification; right-padding would give a meaningless PAD token as the classification representation

5. **Gradient accumulation enables the same effective batch size** as DeBERTa experiments (16 samples), making the LR comparable across model families

### Technical stack

| Component | Library | Purpose |
|---|---|---|
| 4-bit NF4 quantization | `bitsandbytes` | Fit 7B model in 3.5 GB VRAM |
| LoRA adapter injection | `peft` | Train only ~4M parameters |
| Gradient checkpointing | `transformers` + `peft` | Reduce activation memory |
| Linear warmup schedule | `transformers` | Stabilise early adapter updates |
| Class-weighted loss | PyTorch `CrossEntropyLoss` | Compensate for 35/65 imbalance |

### Project ceiling

Mistral-7B LoRA at **macro_f1=0.6553** is the project's best standalone result. The stacking + late fusion approach (Exp 3, DeBERTa OOF) reached 0.6435 — below the LoRA standalone. Adding Mistral's OOF probas to the stacking meta-learner (Lora 1 fusion) reached 0.6492, still below the standalone because fold 1 OOF contamination pulled the meta-learner in a wrong direction. This points to a general principle: **ensemble fusion only helps when all components are well-calibrated**.